# 06 · Which electrodes carry the information?

The same shrinkage LDA, folds and decision rule, trained on one electrode or one region at a time (8 window means per electrode).

*Data: Gao et al. (2014), public Dryad release, doi:10.5061/dryad.2qc64. Own analysis; the plan (folds, training task, decision rule) was fixed before results.*

In [1]:
import os, json, numpy as np, matplotlib.pyplot as plt
# Point GAO_DIR at the "14 electrodes-30 subjects" folder of the Dryad release; GKT_WORK holds all derived files
os.environ.setdefault("GAO_DIR", "./data/14 electrodes-30 subjects/")
os.environ.setdefault("GKT_WORK", "./work/")
from gkt_utils import *
print("data:", GAO_DIR, "| work:", WORK)

data: ./data/14 electrodes-30 subjects/ | work: ./work/


In [2]:
X, G, PID, SES, STIM = load_dataset()
PIDS = sorted(set(PID.tolist()))
print(X.shape, "epochs x channels x samples |", len(PIDS), "people |",
      {k: int((STIM == v).sum()) for k, v in STIM_CODE.items()})

(23388, 12, 128) epochs x channels x samples | 30 people | {'irrelevant': 15583, 'probe': 3890, 'target': 3915}


In [3]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
def run(chs):
    F = window_means(X[:, chs]); s = np.full(len(X), np.nan)
    for k, (pl, ph) in enumerate(PAIRS, 1):
        te = np.isin(PID, [pl, ph]); tr = (~te) & (STIM != 1); y = (STIM[tr] == 2).astype(int)
        sc = StandardScaler().fit(F[tr]); m = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto", priors=[.5, .5]).fit(sc.transform(F[tr]), y)
        s[te] = m.decision_function(sc.transform(F[te]))
    return s
idx = lambda names: [EEG.index(n) for n in names]
SETS = {c: [EEG.index(c)] for c in EEG}
SETS.update({"Frontal (Fp1 Fp2 F3 Fz F4)": idx(["Fp1", "Fp2", "F3", "Fz", "F4"]), "Frontal without Fp1/Fp2": idx(["F3", "Fz", "F4"]),
             "Central (C3 Cz C4)": idx(["C3", "Cz", "C4"]), "Parietal-occipital (P3 Pz P4 Oz)": idx(["P3", "Pz", "P4", "Oz"]),
             "All 12 without Fp1/Fp2": [i for i, c in enumerate(EEG) if c not in ("Fp1", "Fp2")], "All 12 channels": list(range(12))})
OUT, DEC = {}, {}
for name, chs in SETS.items():
    summ, persons = evaluate(run(chs), PID, STIM, n_perm_group=1000)
    OUT[name] = summ; DEC[name] = {p["code"]: p["decision"] for p in persons}
    show(summ, name)

Oz                       trial AUC 0.730 | person AUC 0.911 (perm p = 0.0010) | detected 14/15 | cleared 13/15 | BA 0.900


Pz                       trial AUC 0.815 | person AUC 0.951 (perm p = 0.0020) | detected 14/15 | cleared 13/15 | BA 0.900


P4                       trial AUC 0.793 | person AUC 0.916 (perm p = 0.0020) | detected 14/15 | cleared 11/15 | BA 0.833


C4                       trial AUC 0.732 | person AUC 0.916 (perm p = 0.0020) | detected 13/15 | cleared 12/15 | BA 0.833


P3                       trial AUC 0.811 | person AUC 0.933 (perm p = 0.0020) | detected 14/15 | cleared 13/15 | BA 0.900


...

Frontal without Fp1/Fp2  trial AUC 0.694 | person AUC 0.929 (perm p = 0.0010) | detected 12/15 | cleared 13/15 | BA 0.833


Central (C3 Cz C4)       trial AUC 0.759 | person AUC 0.920 (perm p = 0.0020) | detected 13/15 | cleared 13/15 | BA 0.867


Parietal-occipital (P3 Pz P4 Oz) trial AUC 0.869 | person AUC 0.942 (perm p = 0.0010) | detected 14/15 | cleared 13/15 | BA 0.900


All 12 without Fp1/Fp2   trial AUC 0.897 | person AUC 0.938 (perm p = 0.0010) | detected 14/15 | cleared 13/15 | BA 0.900


All 12 channels          trial AUC 0.896 | person AUC 0.942 (perm p = 0.0010) | detected 14/15 | cleared 13/15 | BA 0.900


In [4]:
import pandas as pd
def metrics(s):
    tp, fp, tn = s["TP"], s["FP"], s["TN"]; prec = tp / (tp + fp); sen = tp / 15
    return dict(trial_AUC=s["trial_auc_target_vs_irrelevant"], person_AUC=s["person_auc"], detected=f"{tp}/15", cleared=f"{tn}/15",
                accuracy=(tp + tn) / 30, precision=prec, sensitivity=sen, specificity=tn / 15, F1=2 * prec * sen / (prec + sen))
pd.DataFrame({k: metrics(v) for k, v in OUT.items()}).T.round(2)

,trial_AUC,person_AUC,detected,cleared,accuracy,precision,sensitivity,specificity,F1
Oz,0.729828,0.911111,14/15,13/15,0.9,0.875,0.933333,0.866667,0.903226
Pz,0.815168,0.951111,14/15,13/15,0.9,0.875,0.933333,0.866667,0.903226
P4,0.793258,0.915556,14/15,11/15,0.833333,0.777778,0.933333,0.733333,0.848485
C4,0.732339,0.915556,13/15,12/15,0.833333,0.8125,0.866667,0.8,0.83871
P3,0.810591,0.933333,14/15,13/15,0.9,0.875,0.933333,0.866667,0.903226
Cz,0.718859,0.897778,12/15,13/15,0.833333,0.857143,0.8,0.866667,0.827586
C3,0.738298,0.915556,11/15,13/15,0.8,0.846154,0.733333,0.866667,0.785714
Fz,0.661934,0.911111,12/15,13/15,0.833333,0.857143,0.8,0.866667,0.827586
F4,0.666431,0.897778,11/15,13/15,0.8,0.846154,0.733333,0.866667,0.785714
F3,0.671805,0.853333,11/15,14/15,0.833333,0.916667,0.733333,0.933333,0.814815


Precision and F1 depend on the whole confusion matrix; one extra error moves sensitivity or specificity by 6.7 points (1/15), but precision and F1 by other amounts.

In [5]:
ref = DEC["All 12 channels"]
for k in ("Parietal-occipital (P3 Pz P4 Oz)", "All 12 without Fp1/Fp2"):
    diff = [c for c in ref if DEC[k][c] != ref[c]]
    print(f"{k}: decisions differing from all 12 channels -> {diff or 'none (identical person by person)'}")

Parietal-occipital (P3 Pz P4 Oz): decisions differing from all 12 channels -> none (identical person by person)
All 12 without Fp1/Fp2: decisions differing from all 12 channels -> none (identical person by person)


**Reading.** Trial AUC is highest at parietal electrodes (Pz, P3, P4) and lowest at Fp1/Fp2. Equal decisions for 30 people do not prove equivalence (trial AUC 0.87 vs 0.90). Scalp data show where information is recorded, not where it is generated.

*The group-level permutation p uses 1,000 shuffles here for speed; the main results use 10,000.*